# authoring/01 — Register a simple agent from scratch

## What this demonstrates

The full agent-authoring flow using the `/api/v1` REST authoring surface:

    POST /api/v1/agents                         # header
    POST /api/v1/agents/{name}/versions          # draft v1.0.0
    POST /api/v1/prompts                         # prompt header
    POST /api/v1/prompts/{name}/versions         # prompt v1.0.0
    POST /api/v1/agents/{name}/versions/{id}/prompts  # assignment
    POST /api/v1/applications/ds_workbench/entities   # map to our app

The new agent stays in `draft` state throughout — no promotion, so the cleanup cell at the end can simply `DELETE` it via the draft-delete endpoint and leave the database exactly as it started.

**Verity capabilities exercised**

- Header + version entity model (agents are named, versions   are immutable once promoted).
- Automatic `{{variable}}` extraction from prompt content.
- Prompt-to-version assignment (many-to-many via   `entity_prompt_assignment`).
- Multi-tenant entity mapping (`application_entity`).


## Prerequisites

- `ds_workbench` application registered (`00_setup.ipynb`).
- At least one inference_config exists (the seeded data   includes several — we pick the first).


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()
v = VerityAPI(application='ds_workbench')

In [ ]:
# Pick an inference_config to reference. Any registered
# config works; we'll use the first one for simplicity.
configs = v.call('list_inference_configs')
assert configs, 'No inference configs registered — Verity seed likely missing.'
cfg = configs[0]
print(f"Using inference_config: {cfg['name']} ({cfg['model_name']})")
INFERENCE_CONFIG_ID = cfg['id']

## Execute

Six REST calls in order: agent header → agent draft version → prompt header → prompt version → prompt assignment → app mapping.


In [ ]:
# 1) Agent header.
AGENT_NAME = 'workbench_demo_agent'
# Delete any leftover from a prior run so this notebook is
# safe to re-execute.
try:
    existing_versions = v.list_agent_versions(AGENT_NAME)
    for ev in existing_versions:
        if ev['lifecycle_state'] == 'draft':
            v.call('delete_agent_version',
                   path_params={'name': AGENT_NAME, 'version_id': ev['id']})
except VerityAPIError:
    pass  # agent doesn't exist yet — first run

try:
    agent_header = v.call('register_agent', json={
        'name': AGENT_NAME,
        'display_name': 'Workbench Demo Agent',
        'description': 'Created from authoring/01 for demo purposes.',
        'purpose': 'Demonstrate the Verity authoring REST surface.',
        'domain': 'demo',
        'materiality_tier': 'low',
        'owner_name': 'ds_workbench',
        'owner_email': None,
        'business_context': None,
        'known_limitations': None,
        'regulatory_notes': None,
    })
    print(f"agent header created: id={agent_header['id']}")
except VerityAPIError as exc:
    if exc.status == 400 and 'duplicate key' in str(exc.detail):
        # Header already exists from a prior run; look it up.
        existing = v.list_agents()
        agent_header = next(a for a in existing if a['name'] == AGENT_NAME)
        print(f"agent header reused: id={agent_header['id']}")
    else:
        raise

In [ ]:
# 2) Agent draft version.
ver = v.call('register_agent_version', path_params={'name': AGENT_NAME}, json={
    'major_version': 1, 'minor_version': 0, 'patch_version': 0,
    'lifecycle_state': 'draft',
    'channel': 'development',
    'inference_config_id': INFERENCE_CONFIG_ID,
    'output_schema': {'type':'object','properties':{'answer':{'type':'string'}}},
    'authority_thresholds': {},
    'mock_mode_enabled': False,
    'decision_log_detail': 'full',
    'developer_name': 'ds_workbench',
    'change_summary': 'Initial version — demo notebook.',
    'change_type': 'initial',
})
AGENT_VERSION_ID = ver['id']
print(f"agent v1.0.0 (draft) id={AGENT_VERSION_ID}")

In [ ]:
# 3) Prompt header.
PROMPT_NAME = 'workbench_demo_system_prompt'
try:
    prompt_header = v.call('register_prompt', json={
        'name': PROMPT_NAME,
        'display_name': 'Workbench demo system prompt',
        'description': 'System message for the workbench demo agent.',
        'primary_entity_type': 'agent',
        'primary_entity_id':   agent_header['id'],
    })
    print(f"prompt header created: id={prompt_header['id']}")
except VerityAPIError as exc:
    if exc.status == 400 and 'duplicate key' in str(exc.detail):
        prompt_header = next(p for p in v.call('list_prompts') if p['name'] == PROMPT_NAME)
        print(f"prompt header reused: id={prompt_header['id']}")
    else:
        raise

# 4) Prompt draft version (template variable {{audience}} will
# be auto-extracted and stored in template_variables).
pv = v.call('register_prompt_version', path_params={'name': PROMPT_NAME}, json={
    'major_version': 1, 'minor_version': 0, 'patch_version': 0,
    'content': 'You are a friendly assistant. Tailor your responses to the {{audience}}.',
    'api_role': 'system',
    'governance_tier': 'behavioural',
    'lifecycle_state': 'draft',
    'change_summary': 'initial',
    'sensitivity_level': 'low',
    'author_name': 'ds_workbench',
})
PROMPT_VERSION_ID = pv['id']
print(f"prompt v1.0.0 (draft) id={PROMPT_VERSION_ID}")

In [ ]:
# 5) Assign the prompt to the agent version.
assignment = v.call(
    'assign_prompt_to_agent',
    path_params={'name': AGENT_NAME, 'version_id': AGENT_VERSION_ID},
    json={
        'prompt_version_id': PROMPT_VERSION_ID,
        'api_role': 'system',
        'governance_tier': 'behavioural',
        'execution_order': 1,
        'is_required': True,
        'condition_logic': None,
    },
)
print(f"prompt assignment id={assignment['id']}")

# 6) Map the agent to the workbench application so our
# cleanup notebook can scope it correctly.
try:
    mapping = v.map_entity(entity_type='agent', entity_id=agent_header['id'])
    print(f"application_entity mapping id={mapping['id']}")
except VerityAPIError as exc:
    # POST application_entity is ON CONFLICT DO NOTHING, but
    # the API wrapper treats empty rows differently; ignore.
    print(f"mapping skipped or already present: {exc.detail}")

## Review results

Resolve the full config for our new draft agent and render it the same way the admin UI would — the prompt assignment should flow through automatically.


In [ ]:
config = v.get_agent_config(AGENT_NAME, version_id=AGENT_VERSION_ID)
render_detail(
    AGENT_NAME,
    subtitle=f"v{config['version_label']}",
    header_badges=[
        (config.get('lifecycle_state','?'), config.get('lifecycle_state','')),
        (config.get('materiality_tier','?'), config.get('materiality_tier','')),
    ],
    sections=[
        {'title':'Inference config','fields':[
            ('Name',        config['inference_config'].get('name')),
            ('Model',       config['inference_config'].get('model_name')),
            ('Temperature', config['inference_config'].get('temperature')),
        ]},
        {'title':f"Prompts ({len(config.get('prompts') or [])})",
         'table':{
             'columns':[
                 ('prompt_name','Prompt'),
                 ('version_number','Version'),
                 ('api_role','Role','neutral'),
                 ('governance_tier','Tier'),
             ],
             'rows': config.get('prompts') or [],
         }},
    ],
)

In [ ]:
# Version lineage — shows the 1.0.0 draft hanging off the
# agent header with nothing promoted above it yet. As the
# author iterates (clone → PATCH → promote), more nodes
# and clone-edges appear.
from utility.visualizations import version_lineage_graph
versions = v.list_agent_versions(AGENT_NAME)
version_lineage_graph(versions)

## Cleanup

Delete the draft version + entity mapping so this notebook is idempotent. The agent header row remains, which is fine — re-running the notebook will skip the duplicate-key error and reuse it.


In [ ]:
try:
    v.call('delete_agent_version',
           path_params={'name': AGENT_NAME, 'version_id': AGENT_VERSION_ID})
    print('draft version deleted')
except VerityAPIError as exc:
    print(f'draft delete failed: {exc.detail}')

try:
    v.call('unmap_entity',
           path_params={'name': 'ds_workbench',
                        'entity_type': 'agent',
                        'entity_id': agent_header['id']})
    print('app entity mapping removed')
except VerityAPIError as exc:
    if exc.status == 404:
        print('mapping was already gone')
    else:
        raise

---

Note: `delete_prompt_version` is only valid on drafts, but our draft prompt is still assigned to the agent (if the agent draft is gone first, the prompt is orphaned and can be removed separately). Re-running this notebook handles the reuse path automatically.

Move on to **`authoring/02_clone_and_edit_draft.ipynb`** to see the clone-and-edit workflow against an existing champion.
